# Emotion Lexicon Layer — From Physiology to Emotion Tokens

---

## Goal

In the previous stage, I converted physiological signals into structured
HRV feature vectors. In this notebook, the goal is to translate physiological patterns into **symbolic emotion tokens**. Instead of treating emotion as a final label, Echo introduces an intermediate representation, a lexicon that allows emotional states to be combined and interpreted in later layers.

This marks the transition from **physiology** to **emotional language**.

---

### Outputs of this notebook

- `lexicon_matrix.csv`  
  A window level table that attaches an emotion token to each HRV segment.

- `emotion_lexicon.json`  
  A dictionary that defines each token in human readable form and stores
  metadata for later semantic and musical layers.

---

## Why an emotion lexicon?

Most affective computing systems directly map physiological signals
to emotion labels. However, labels are static and final. They do not allow
further composition or interpretation.

Echo takes a different approach.

Rather than predicting emotions, the system **constructs a symbolic vocabulary**
that can later be translated, combined, and expressed. In this sense, emotion tokens are not outcomes. They are building blocks.

---

## Emotion tokens are not labels

Emotion tokens in Echo are intentionally different from traditional emotion
categories. A label such as “stress” or “calm” represents a final decision. A token, on the other hand, functions more like a word in a language. It does not fully describe meaning by itself, but gains meaning through context, and combination with other tokens.

This allows emotional experience to be represented as something dynamic,
rather than a single fixed prediction.

| Linguistics concept | Echo counterpart |
|--------------------|------------------|
| Word | Emotion token |
| Morphology | Token schema |
| Syntax | Token transitions |
| Semantics | Text interpretation |
| Prosody | Musical expression |


---

## Token structure and emotional morphology

Each emotion token follows a structured schema 
[E_State_Arousal_Index]

For example:

- `⟦E_Calm_Low_01⟧`
- `⟦E_Tense_High_01⟧`

This design is inspired by linguistic morphology, where meaning is constructed
by combining smaller semantic components.

Here I designed 3 parts:

- **State** captures variability patterns related to physiological stability  
- **Arousal** reflects activation level inferred from heart rate  
- **Index** allows multiple variants within the same category  

Instead of treating emotion as a single unit, Echo represents it as a
structured combination of interpretable parts.

---

## Emotional grammar through time

Emotion rarely appears as a single isolated moment. It unfolds over time. By assigning one token to each temporal window, Echo represents emotional experience as a **sequence of tokens**.

For example, [E_Calm_Low] -> [E_Neutral_Mid] -> [E_Tense_High]

This sequential structure makes it possible to later describe emotional patterns such as transitions, persistence, escalation, recovery etc.

In this way, emotional dynamics begin to resemble a simple grammar, just like a language we are speaking. Emotional dynamics can become patterns of change that carry meaning.

---

## 1. Set up

In this section, the HRV feature table generated in Stage 3 is loaded. This table serves as the physiological foundation for token construction.


In [2]:
processed_path = Path("../results")
features_file = processed_path / "features_hrv.csv"

df = pd.read_csv(features_file)
df.head()


,subject,window_id,t_start,t_end,MeanNN,SDNN,RMSSD,pNN50,HR_mean,LF_HF,quality_score,n_beats
0,S2,0,1.725714,61.725714,0.790207,0.085074,0.043753,0.213333,76.791505,NaN,1.0,76
1,S2,1,31.725714,91.725714,0.759747,0.060547,0.040032,0.166667,79.465489,NaN,1.0,79
2,S2,2,61.725714,121.725714,0.790376,0.051677,0.040975,0.200000,76.237465,NaN,1.0,76
3,S2,3,91.725714,151.725714,0.815077,0.053393,0.044399,0.315068,73.921045,NaN,1.0,74
4,S2,4,121.725714,181.725714,0.807600,0.066821,0.049256,0.337838,74.806137,NaN,1.0,75


---

## 2. Define mapping strategy

The lexicon in version 1.0 focuses on interpretability and conceptual clarity. The goal at this stage is to establish a clear and interpretable mapping foundation for the emotional language system.


Physiological features are mapped into symbolic states using simple, transparent rules that can later be replaced or refined.

---

## 3. Build helper functions

Helper functions are defined to:

- compute robust thresholds from the dataset  
- assign physiological states from HRV variability  
- assign arousal levels from heart rate  
- construct standardized emotion tokens  

These functions ensure that token generation is consistent across subjects
and time windows.

In [5]:
def robust_thresholds(series: pd.Series):
    """Return low and high thresholds using quantiles."""
    low = series.quantile(0.33)    # the low threshold is the 33rd percentile because we want to capture the lower third of the data to identify tense states
    high = series.quantile(0.66)   # the high threshold is the 66th percentile because we want to capture the upper third of the data to identify calm states
    return float(low), float(high)
# the reason we use third quantiles instead of quartiles is to have a more balanced division of the data into three states: calm, neutral, and tense

rmssd_low, rmssd_high = robust_thresholds(df["RMSSD"])   # this is to get the thresholds for RMSSD values so that we can classify the states based on vagal activity
hr_low, hr_high = robust_thresholds(df["HR_mean"])   # this is to get the thresholds for HR_mean values so that we can classify the arousal levels

rmssd_low, rmssd_high, hr_low, hr_high


def assign_state_from_rmssd(rmssd: float, low: float, high: float) -> str:
    """
    RMSSD is used as a simple proxy for vagal activity / calmness.
    - high RMSSD -> Calm
    - low RMSSD  -> Tense
    - middle     -> Neutral
    """
    if np.isnan(rmssd):
        return "Unknown"
    if rmssd >= high:
        return "Calm"
    if rmssd <= low:
        return "Tense"
    return "Neutral"


def assign_arousal_from_hr(hr: float, low: float, high: float) -> str:
    """
    HR_mean is used as a proxy for arousal.
    """
    if np.isnan(hr):     # this is the case when the HR value is not available
        return "Unknown"
    if hr >= high:      # this is the case when the HR value is above the high threshold, which means that the person is in a high arousal state
        return "High"
    if hr <= low:        # when it's below the low threshold, indicating a low arousal state
        return "Low"
    return "Mid"     # a mid arousal state when the HR value is between the low and high thresholds, meaning that the person is in a moderate arousal state


def build_token(state: str, arousal: str, idx: int = 1) -> str:
    """
    Token schema:
    ⟦E_<State>_<Arousal>_<Index>⟧
    """
    return f"⟦E_{state}_{arousal}_{idx:02d}⟧"   # this is the format for the token designed by me to represent emotional states!!


---

## 4. Generate lexicon matrix

In this step, emotion tokens are assigned to every HRV window.

The resulting table contains

- physiological features  
- derived state and arousal components  
- final emotion token  

Each row now represents a short emotional moment expressed in symbolic form. This table forms the backbone of Echo’s emotional language layer! ^^

In [7]:
df = df.copy()     # this is to avoid modifying the original dataframe

df["state"] = df["RMSSD"].apply(lambda x: assign_state_from_rmssd(x, rmssd_low, rmssd_high))    # assign emotional states based on RMSSD values
df["arousal"] = df["HR_mean"].apply(lambda x: assign_arousal_from_hr(x, hr_low, hr_high))       # assign arousal levels based on HR_mean values
df["emotion_token"] = df.apply(lambda r: build_token(r["state"], r["arousal"], 1), axis=1)      # build emotion tokens for each row in the dataframe

# this step is to create a lexicon matrix
lexicon_matrix = df[
    ["subject", "window_id", "t_start", "t_end",
     "MeanNN", "SDNN", "RMSSD", "pNN50", "HR_mean", "LF_HF",
     "quality_score", "n_beats",
     "state", "arousal", "emotion_token"]
].copy()

lexicon_matrix.head()


lexicon_matrix_file = processed_path / "lexicon_matrix.csv"
lexicon_matrix.to_csv(lexicon_matrix_file, index=False)

print("Saved:", lexicon_matrix_file)
print("Unique tokens:", lexicon_matrix["emotion_token"].nunique())

Saved: ../results/lexicon_matrix.csv
Unique tokens: 9


---

## 5. Build emotion lexicon dictionary

Finally, a dictionary is constructed to define each emotion token. This is one of the most exciting parts!

For every unique token, the lexicon stores

- its physiological interpretation  
- its valence category  
- a short human readable description  
- optional default parameters for expressive layers  

This dictionary allows tokens to be translated into language and music in later stages.

Notice, this lexicon is intentionally minimal in version 1.0 and is designed to grow alongside the system.


In [10]:
def token_metadata(state: str, arousal: str):
    """
    Generate minimal metadata for each emotion token (v1 lexicon).

    This function intentionally uses simple, interpretable rules rather than
    learned models. The goal at this stage is conceptual clarity, not optimization.

    All mappings defined here act as transparent placeholders that can be
    refined or replaced in later versions of Echo.
    """

    # Here we introduce a very coarse emotional polarity.
    # This is  a functional abstraction that allows later semantic and musical layers to reason about directionality.
    
    # Calm -> Positive: associated with physiological stability and parasympathetic dominance
    # Tense -> Negative: associated with elevated arousal and stress related variability
    # Neutral -> Neutral: serves as a transitional or baseline state
    # Other -> Unknown: preserves uncertainty 
    
    # This design keeps early emotional representations simple and interpretable

    if state == "Calm":
        valence = "Positive"
    elif state == "Tense":
        valence = "Negative"
    elif state == "Neutral":
        valence = "Neutral"
    else:
        valence = "Unknown"

    
    # These values serve as structural anchors for the later music layer.
    
    # Tempo mapping
    # Low arousal -> slower tempo (80 bpm): reflects physiological calm and reduced activation
    # Mid arousal -> moderate tempo (110 bpm): represents transitional or neutral engagement
    # High arousal -> faster tempo (130 bpm): reflects heightened physiological activation
    
    # These values' relative ordering matters more than absolute magnitude.
   
    tempo = (
        80 if arousal == "Low"
        else 110 if arousal == "Mid"
        else 130
    )

    
    # Mode mapping
    # Positive valence -> major
    # Negative valence -> minor
    # Neutral / unknown -> neutral
    
    # This mirrors common conventions in music perception research, while remaining deliberately simple and interpretable.
    
    mode = (
        "major" if valence == "Positive"
        else "minor" if valence == "Negative"
        else "neutral"
    )

    # Intensity mapping 
    # Arousal level is used as a proxy for expressive intensity.
    # Low -> soft
    # Mid -> medium
    # High -> strong
    
    # This allows physiological activation to later modulate expressive strength in the music layer. Expressive intensity is distinct from tempo and mode, providing an additional dimension of musical expression.
    
    intensity = (
        "soft" if arousal == "Low"
        else "medium" if arousal == "Mid"
        else "strong"
    )

    music = {
        "tempo_bpm": tempo,
        "mode": mode,
        "intensity": intensity
    }

    return valence, music

# this is to build emotion lexicon dictionary and collect unique emotion tokens from the lexicon matrix
unique_tokens = sorted(lexicon_matrix["emotion_token"].unique())

lexicon = {}

for tok in unique_tokens:
    # Token format:
    # ⟦E_State_Arousal_01⟧
    core = tok.strip("⟦⟧")
    _, state, arousal, idx = core.split("_")

    # this is to retrieve metadata based on the mappings I made
    valence, music = token_metadata(state, arousal)

    # this is to store structured definition for each token
    lexicon[tok] = {
        "state": state,
        "arousal": arousal,
        "valence": valence,
        "definition": f"{state} state with {arousal.lower()} arousal (prototype rule-based mapping).",
        "music_defaults": music
    }

# this is to save emotion lexicon
lexicon_file = processed_path / "emotion_lexicon.json"

with open(lexicon_file, "w") as f:
    json.dump(lexicon, f, indent=2)

print("Saved:", lexicon_file)

# quick sanity check
list(lexicon.items())[:2]

Saved: ../results/emotion_lexicon.json


[('⟦E_Calm_High_01⟧',
  {'state': 'Calm',
   'arousal': 'High',
   'valence': 'Positive',
   'definition': 'Calm state with high arousal (prototype rule-based mapping).',
   'music_defaults': {'tempo_bpm': 130,
    'mode': 'major',
    'intensity': 'strong'}}),
 ('⟦E_Calm_Low_01⟧',
  {'state': 'Calm',
   'arousal': 'Low',
   'valence': 'Positive',
   'definition': 'Calm state with low arousal (prototype rule-based mapping).',
   'music_defaults': {'tempo_bpm': 80, 'mode': 'major', 'intensity': 'soft'}})]

---

## Summary

At the end of this notebook, physiological signals have been transformed into a symbolic emotional vocabulary.

Emotion now is able to be treated as a prediction target, but as a structured representation that can be composed and expressed. This lexicon serves as the conceptual bridge between body signals and higher level emotional meaning in Echo.